# Insurance Claims Recovery Data Audit

## Objective

The objective of this notebook is to assess the quality,
completeness, consistency, and reliability of the insurance
claims dataset before any analysis or dashboard development.

The audit covers:

- Dataset Overview
- Missing Values
- Duplicate Analysis
- Data Types
- Business Rule Validation
- Outlier Detection


## Imports and settings

In [2]:
#Data manipulation 
import pandas as pd
import numpy as np

#Data Visualization
import matplotlib.pyplot as plt
import plotly.express as px
from src.config import DATA_PATH, LOCATION_MAPPING_PATH

#display settings
pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows", 100)

## Load data set

In [3]:
df= pd.read_excel(DATA_PATH)
df.head()

,Claim ID,Claim Number,Original Amount,Approved Discount %,Recovery Amount,Remaining Amount,Collected Amount,Tp Responsiblity,Debtor Name,Debtor Number,Debtor Iqama,Driver Name,Driver Number,Driver Iqama,Owner Type,Accident Date,RCP Date,Accident Location,Debtor Type,Status,Reason,Officer,ASCIC Remark,Latest Remark,Unnamed: 24,Unnamed: 25,Recovery Reason,Legal Remark
0,ASCIC03267,C/ERO1/2026/TPSD/0001285,3400.0,NaN,3400.0,3400.0,0.0,0,لجى هادي لحيس العنزي,966506407923,1144521935,NaN,NaN,NaN,Individual,2026-07-14 00:00:00,26-07-19,Riyadh,insured,NaN,NaN,MAJED,لا يملك رخصة,NaN,NaN,NaN,لا يملك رخصة قيادة,No legal remarks
1,ASCIC03268,C/CRO1/2026/CMC/0010021,6310.2,NaN,6310.2,6310.2,0.0,0.75,منصور سالم سطام الشمري,966506116165,1111663207,NaN,NaN,NaN,Individual,2026-04-27 00:00:00,26-07-19,Riyadh,third_party,Approved معلق,NaN,MAJED,"استرداد 6,310,20 ريال من المبلغ الزائد الذي تم...",NaN,NaN,NaN,NaN,No legal remarks
2,ASCIC03266,C/CRO1/2026/MTP/0007280,1382.0,NaN,1382.0,1382.0,0.0,0,نهاد علي المطر,966502039692,1013589880,NaN,NaN,NaN,Individual,2026-05-06 00:00:00,26-07-16,NaN,third_party,Approved معلق,NaN,MAJED,استرداد مبلغ التعويض الزائد بعد خطأ في التقدير...,NaN,NaN,NaN,NaN,No legal remarks
3,ASCIC03265,C/CRO1/2026/CMC/0005730,9972.0,NaN,9972.0,9972.0,0.0,0,ﻣﻨﻴﺮﻩ ﺣﻤﺪ ﻓﻴﺼﻞ ﺍﻟﻌﺠﻤﻲ,966595266069,1073664979,مبارك فهيد بن مبارك,9.665953e+11,1.107695e+09,Individual,2026-03-08 00:00:00,26-07-16,الاحساء,insured,Approved معلق,NaN,Munirah,انتهاء رخصة قيادة,NaN,NaN,NaN,أنتهاء تاريخ رخصة القيادة,No legal remarks
4,ASCIC03248,C/CRO1/2026/PTOA/0013353,4726.8,NaN,4726.8,4726.8,0.0,0,خالد بن فهيد بن عائض السهلي,966503339905,1018847036,زهيب أحمد سرور,9.665400e+11,2.616614e+09,Individual,2026-06-25 00:00:00,26-07-16,JEDDAH,insured,Approved معلق,NaN,Maha Mohammed,عكس السير,NaN,NaN,NaN,عكس اتجاه السير,No legal remarks


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3267 entries, 0 to 3266
Data columns (total 28 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Claim ID             3267 non-null   object 
 1   Claim Number         3267 non-null   object 
 2   Original Amount      3267 non-null   float64
 3   Approved Discount %  37 non-null     float64
 4   Recovery Amount      3267 non-null   float64
 5   Remaining Amount     3267 non-null   float64
 6   Collected Amount     3267 non-null   float64
 7   Tp Responsiblity     2664 non-null   object 
 8   Debtor Name          3265 non-null   object 
 9   Debtor Number        3267 non-null   int64  
 10  Debtor Iqama         3267 non-null   object 
 11  Driver Name          940 non-null    object 
 12  Driver Number        851 non-null    float64
 13  Driver Iqama         821 non-null    float64
 14  Owner Type           3267 non-null   object 
 15  Accident Date        2745 non-null   o

## Key Observations

- The dataset contains **3,267 records** and **28 columns**.
- Most columns are categorical (`object`), while 8 columns are numeric.
- Key financial columns (Original Amount, Recovery Amount, Remaining Amount, Collected Amount) contain no missing values.
- Accident Date and RCP Date are stored as text and will require conversion to datetime.
- Driver-related columns have substantial missing values and require business validation before any cleaning decisions.
- Approved Discount % contains values for only about 1% of records, indicating that it may be an optional or specialized field.
- Two unnamed columns appear to be artifacts from the Excel export and will be investigated further.

In [5]:
print(f"Number of rows in the dataset: {df.shape[0]:,}")
print(f"Number of cols in the dataset: {df.shape[1]}")

Number of rows in the dataset: 3,267
Number of cols in the dataset: 28


In [6]:
for cols in df.columns:
    print(cols)

Claim ID
Claim Number
Original Amount
Approved Discount %
Recovery Amount
Remaining Amount
Collected Amount
Tp Responsiblity
Debtor Name
Debtor Number
Debtor Iqama
Driver Name
Driver Number
Driver Iqama
Owner Type
Accident Date
RCP Date
Accident Location
Debtor Type
Status
Reason
Officer
ASCIC Remark
Latest Remark
Unnamed: 24
Unnamed: 25
Recovery Reason
Legal Remark


In [7]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Claim ID,3267,3267,ASCIC03267,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Claim Number,3267,3247,C/CRO1/2025/PTRA/0001701,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Original Amount,3267.0,NaN,NaN,NaN,10970.19566,16634.77662,1.0,3000.0,7984.7,13096.195,325230.0
Approved Discount %,37.0,NaN,NaN,NaN,0.169449,0.105454,0.0,0.1,0.15,0.2,0.5
Recovery Amount,3267.0,NaN,NaN,NaN,10937.824405,16598.955516,1.0,2998.0,7969.0,13050.0,325230.0
Remaining Amount,3267.0,NaN,NaN,NaN,10482.225605,16602.753846,0.0,2500.0,7719.0,12748.0,325230.0
Collected Amount,3267.0,NaN,NaN,NaN,454.98183,2677.255921,0.0,0.0,0.0,0.0,60630.0
Tp Responsiblity,2664.0,6.0,0.0,2205.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Debtor Name,3265,3209,Saudi National Bank,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Debtor Number,3267.0,NaN,NaN,NaN,968021817669.019531,155836225171.391632,966.0,966522354526.0,966549407800.0,966560202245.0,9660500618373.0


In [8]:
df.sample(5)

,Claim ID,Claim Number,Original Amount,Approved Discount %,Recovery Amount,Remaining Amount,Collected Amount,Tp Responsiblity,Debtor Name,Debtor Number,Debtor Iqama,Driver Name,Driver Number,Driver Iqama,Owner Type,Accident Date,RCP Date,Accident Location,Debtor Type,Status,Reason,Officer,ASCIC Remark,Latest Remark,Unnamed: 24,Unnamed: 25,Recovery Reason,Legal Remark
2201,ASCIC01061,V/TP/RYD/2016/81612,78930.0,NaN,78930.0,78930.0,0.0,0,خضراء سلامه الحويطي,966504654546,1067992964,NaN,NaN,NaN,Not Defined,2016-11-07 00:00:00,26-01-26,Jeddah,insured,Transfer To Law رفض السداد,NaN,NaN,صدم و هروب من موقع الحادث - لا يملك رخصة قيادة,NaN,NaN,NaN,صدم و هروب من موقع الحادث,2026-04-28: تقادم لا يمكن تقديم دعوى (by Muteb...
1053,ASCIC02215,C/CRO1/2025/PTRA/0000098,4923.0,NaN,4923.0,4923.0,0.0,0,ريم عبدالله سالم المالكي,966548911116,1064996943,جيكول خان مير خان,9.665555e+11,2.458296e+09,Individual,2024-12-20 00:00:00,26-04-19,مكة المكرمة,insured,Approved معلق,NaN,M.ALBABTAIN,انتهاء الرخصة,NaN,NaN,NaN,أنتهاء تاريخ رخصة القيادة,No legal remarks
214,ASCIC03068,C/WRO1/2026/CV/0000115,10530.5,NaN,10530.5,10530.5,0.0,0,عليش أوسين,966571605221,2609983149,NaN,NaN,NaN,Individual,2026-03-17 00:00:00,26-06-14,Riyadh,insured,Approved معلق,NaN,Jasim,نوع الرخصه,NaN,NaN,NaN,نوع الرخصة لا يخوله بقيادة المركبة,No legal remarks
1926,ASCIC01405,V/TP/RYD/2015/56648,13150.0,NaN,13150.0,13150.0,0.0,0,ايمان خليل حماد,966530636078,1118766862,NaN,NaN,NaN,Not Defined,2015-05-02 00:00:00,26-01-26,Al-Jouf,insured,Approved معلق,NaN,NaN,لا يملك رخصة قيادة,NaN,NaN,NaN,لا يملك رخصة قيادة,No legal remarks
1073,ASCIC02195,C/CRO1/2024/CV/0006376,2230.0,NaN,2230.0,2230.0,0.0,0,محمد جبريل,966548794838,2502398346,NaN,NaN,NaN,Individual,2024-09-12 00:00:00,26-04-16,الظهران,insured,Approved معلق,NaN,Sultan Alharbi,رخصة السائق لا تخول,NaN,NaN,NaN,نوع الرخصة لا يخوله بقيادة المركبة,No legal remarks


##### The Accident Date and RCP Date columns are currently stored as text (`object`) instead of datetime objects. These columns should be converted to datetime format before performing time-series analysis.

In [9]:
pd.to_datetime(df["Accident Date"], errors="coerce")

0      2026-07-14
1      2026-04-27
2      2026-05-06
3      2026-03-08
4      2026-06-25
          ...    
3262   2024-07-19
3263   2024-08-11
3264   2022-03-22
3265   2025-03-13
3266          NaT
Name: Accident Date, Length: 3267, dtype: datetime64[ns]

In [10]:
pd.to_datetime(df["RCP Date"], errors="coerce")

C:\Users\Subhan\AppData\Local\Temp\ipykernel_40884\1985303089.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(df["RCP Date"], errors="coerce")


0      2019-07-26
1      2019-07-26
2      2016-07-26
3      2016-07-26
4      2016-07-26
          ...    
3262   2017-04-25
3263   2017-04-25
3264   2017-04-25
3265   2017-04-25
3266   2016-04-25
Name: RCP Date, Length: 3267, dtype: datetime64[ns]

## Missing Value Analysis

In [11]:
missing_df = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing %": (df.isnull().sum() / len(df) * 100).round(2)
})

missing_df = missing_df[missing_df["Missing Values"] > 0]
missing_df = missing_df.sort_values("Missing %", ascending=False)

missing_df  

,Missing Values,Missing %
Unnamed: 24,3249,99.45
Latest Remark,3245,99.33
Unnamed: 25,3245,99.33
Approved Discount %,3230,98.87
Reason,2858,87.48
Driver Iqama,2446,74.87
Driver Number,2416,73.95
Driver Name,2327,71.23
ASCIC Remark,665,20.36
Officer,613,18.76


## Duplicate Analysis

In [12]:
print(f"Number of duplicated rows: {df.duplicated().sum()}")

Number of duplicated rows: 0


In [13]:
print(df["Claim ID"].duplicated().sum())
print(df["Claim Number"].duplicated().sum())

0
20


In [14]:
duplicate_claims = (
    df[df["Claim Number"].duplicated(keep=False)]
    .sort_values("Claim Number")
)

duplicate_claims

,Claim ID,Claim Number,Original Amount,Approved Discount %,Recovery Amount,Remaining Amount,Collected Amount,Tp Responsiblity,Debtor Name,Debtor Number,Debtor Iqama,Driver Name,Driver Number,Driver Iqama,Owner Type,Accident Date,RCP Date,Accident Location,Debtor Type,Status,Reason,Officer,ASCIC Remark,Latest Remark,Unnamed: 24,Unnamed: 25,Recovery Reason,Legal Remark
1120,ASCIC02148,C/CRO1/2024/PTRA/0007227,3709.95,NaN,3709.95,3709.95,0.0,0,ابراهيم خان ميان زرين,966596007437,2506478771,NaN,NaN,NaN,Individual,2024-12-29 00:00:00,26-04-06,Riyadh,insured,NaN,Revert,MAJED,عكس اتجاه السير,NaN,NaN,NaN,عكس اتجاه السير,No legal remarks
1072,ASCIC02196,C/CRO1/2024/PTRA/0007227,3710.00,NaN,3710.00,3710.00,0.0,0,ابراهيم خان ميان زرين,966549171198,2506478771,NaN,NaN,NaN,Individual,2024-12-29 00:00:00,26-04-16,الرياض,insured,Approved معلق,NaN,Sultan Alharbi,عكس اتجاه السير,NaN,NaN,NaN,عكس اتجاه السير,No legal remarks
1038,ASCIC02230,C/CRO1/2025/CMC/0024244,95358.00,NaN,95358.00,95358.00,0.0,0,ساجر زامن رادا كام,966583230415,2529733434,NaN,NaN,NaN,Individual,2025-12-12 00:00:00,26-04-19,الرياض,insured,Approved معلق,NaN,NAIF,نوع الرخصة لايخول بالقيادة,NaN,NaN,NaN,نوع الرخصة لا يخوله بقيادة المركبة,No legal remarks
998,ASCIC02270,C/CRO1/2025/CMC/0024244,230230.00,NaN,230230.00,230230.00,0.0,0,ساجر زامن رادا كام,966583230415,2529733434,NaN,NaN,NaN,Individual,2025-12-12 00:00:00,26-04-20,الرياض,insured,Approved معلق,NaN,NAIF,نوع الرخصة لايخول بالقيادة,NaN,NaN,NaN,حالات حق الرجوع / الافراد,No legal remarks
3253,ASCIC022,C/CRO1/2025/PTOA/0001076,20230.00,NaN,20230.00,20230.00,0.0,NaN,مطلق محمد القحطاني,966501965969,1141374031,NaN,NaN,NaN,Individual,2025-01-05 00:00:00,25-04-29,أبها,third_party,Transfer To Law رفض السداد,NaN,MAJED,NaN,NaN,NaN,NaN,NaN,2026-01-28: رسالة التبليغ غير مرفقه (by Muteb ...
999,ASCIC02269,C/CRO1/2025/PTOA/0001076,79500.00,NaN,79500.00,79500.00,0.0,0,مطلق محمد القحطاني,966501965969,1141374031,سعود القحطاني,9.665020e+11,1.096248e+09,Individual,2025-01-05 00:00:00,26-04-20,أبها,insured,Approved معلق,NaN,NAIF,انتهاء الرخصة -اصابة,NaN,NaN,NaN,أنتهاء تاريخ رخصة القيادة,No legal remarks
700,ASCIC02568,C/CRO1/2025/PTRA/0001548,421.00,NaN,421.00,0.00,421.0,1,غلام دست غير غلام,966539624427,2588354148,NaN,NaN,NaN,Individual,2025-01-28 00:00:00,26-04-23,حائل,insured,Collected تم السداد - كامل,NaN,Jasim,لا يملك رخصة,NaN,NaN,NaN,لا يملك رخصة قيادة,No legal remarks
418,ASCIC02850,C/CRO1/2025/PTRA/0001548,227.00,NaN,227.00,0.00,227.0,0,GHULAM DASTGEER GHULAM RASOOL,966539624427,2588354148,NaN,NaN,NaN,Individual,2025-01-28 00:00:00,26-05-04,حائل,insured,Collected تم السداد - كامل,NaN,MAJED,تم تحصيل مبلغ 421 من أصل 647.80 ريال والمتبقي ...,NaN,NaN,NaN,لا يملك رخصة قيادة,No legal remarks
905,ASCIC02363,C/CRO1/2025/PTRA/0001701,18379.00,NaN,18379.00,18379.00,0.0,0,شهاب الدين احمد حامد شحاتل,966523480066,2373480066,شهاب الدين احمد حامد شحاتل,9.665235e+11,2.373480e+09,Individual,2025-01-30 00:00:00,26-04-21,مكة المكرمة,insured,Closed حذف المسترد,تم تجديد الرخصة في المدة النظامية,NAIF,انتهاء الرخصة,NaN,NaN,NaN,أنتهاء تاريخ رخصة القيادة,No legal remarks
902,ASCIC02366,C/CRO1/2025/PTRA/0001701,12000.00,NaN,12000.00,12000.00,0.0,0,شهاب الدين احمد حامد شحاتل,966523480066,2373480066,شهاب الدين احمد حامد شحاتل,9.665235e+11,2.373480e+09,Individual,2025-01-30 00:00:00,26-04-21,مكة المكرمة,insured,Closed حذف المسترد,تم تجديد الرخصة في المدة النظامية,NAIF,انتهاء الرخصة,NaN,NaN,NaN,أنتهاء تاريخ رخصة القيادة,No legal remarks


## Business Rules

In [15]:
#rule 1
rule1=df[df["Recovery Amount"] > df["Original Amount"]]
rule1

,Claim ID,Claim Number,Original Amount,Approved Discount %,Recovery Amount,Remaining Amount,Collected Amount,Tp Responsiblity,Debtor Name,Debtor Number,Debtor Iqama,Driver Name,Driver Number,Driver Iqama,Owner Type,Accident Date,RCP Date,Accident Location,Debtor Type,Status,Reason,Officer,ASCIC Remark,Latest Remark,Unnamed: 24,Unnamed: 25,Recovery Reason,Legal Remark


In [16]:
#rule 2
rule2=df[df["Collected Amount"] > df["Recovery Amount"]]
rule2

,Claim ID,Claim Number,Original Amount,Approved Discount %,Recovery Amount,Remaining Amount,Collected Amount,Tp Responsiblity,Debtor Name,Debtor Number,Debtor Iqama,Driver Name,Driver Number,Driver Iqama,Owner Type,Accident Date,RCP Date,Accident Location,Debtor Type,Status,Reason,Officer,ASCIC Remark,Latest Remark,Unnamed: 24,Unnamed: 25,Recovery Reason,Legal Remark


In [17]:
#rule 3
rule3=df[df["Remaining Amount"] != (df["Recovery Amount"].round(2) - df["Collected Amount"].round(2))]
rule3

,Claim ID,Claim Number,Original Amount,Approved Discount %,Recovery Amount,Remaining Amount,Collected Amount,Tp Responsiblity,Debtor Name,Debtor Number,Debtor Iqama,Driver Name,Driver Number,Driver Iqama,Owner Type,Accident Date,RCP Date,Accident Location,Debtor Type,Status,Reason,Officer,ASCIC Remark,Latest Remark,Unnamed: 24,Unnamed: 25,Recovery Reason,Legal Remark
210,ASCIC03064,C/CRO1/2026/MTP/0003270,7745.10,NaN,7745.10,7745.0,0.0,0,محمد عدنان محمد البخاري,966505677550,1080792839,على صديق عثمان يوسف,9.665342e+11,2.492336e+09,Individual,2026-04-10 00:00:00,26-06-14,JEDDAH,insured,Partial Payment تم السداد - جزئي,حسب طلب العميل,Maha Mohammed,عكس السير,NaN,NaN,NaN,عكس اتجاه السير,No legal remarks
313,ASCIC02954,C/CRO1/2026/PTRA/0002664,14439.95,NaN,14439.95,14436.0,0.0,0,ريم بدر الدين - الحاج علي,966562562730,2270720556,طلال بدر الدين,9.665708e+11,4.838466e+09,Individual,2026-03-19 00:00:00,26-05-14,Jeddah,insured,Partial Payment تم السداد - جزئي,وعد بالسداد بتاريخ : 25/07/2026 على 6 دفعات,Jasim,السائق لايملك رخصة,NaN,NaN,NaN,لا يملك رخصة قيادة,No legal remarks
873,ASCIC02383,C/CRO1/2026/PTOA/0003950,9134.85,NaN,9134.85,4134.0,5000.0,0,مبارك محمد مرعي الشهراني,966509222052,1014116410,NaN,NaN,NaN,Individual,2026-01-16 00:00:00,26-04-21,Riyadh,insured,Partial Payment تم السداد - جزئي,حسب طلب العميل,MAJED,أنتهاء تاريخ رخصة القيادة,NaN,NaN,NaN,أنتهاء تاريخ رخصة القيادة,No legal remarks
2244,ASCIC01055,C/CRO1/2025/PCRR/0023438,19752.80,NaN,19752.80,16002.0,3750.0,0,محمد عبدالعزيز بن جبر,966549651521,1072425372,NaN,NaN,NaN,Not Defined,2025-11-19 00:00:00,26-01-26,Riyadh,insured,Partial Payment تم السداد - جزئي,عن طريق مدلين لتحصيل الديون,MAJED,تجاوز الإشارة الحمراء,NaN,NaN,NaN,تجاوز الإشارة الحمراء,No legal remarks
2507,ASCIC0771,C/CRO1/2025/PTRA/0016535,14135.00,NaN,14135.00,3531.0,10598.0,%,زينب حسين بن محمد البناي,966564292592,1023076829,NaN,NaN,NaN,Not Defined,2025-10-19 00:00:00,25-12-02,الاحساء,insured,Partial Payment تم السداد - جزئي,NaN,MAJED,لا يملك رخصة قيادة,NaN,NaN,NaN,لا يملك رخصة قيادة,No legal remarks
2581,ASCIC0685,C/ERO1/2025/CMC/0001451,3771.35,NaN,3771.35,1771.0,2000.0,NaN,ايهاب السيد احمد رستم,966546152692,2545554954,NaN,NaN,NaN,Not Defined,2025-07-27 13:44:00,25-11-06,Al Dammam,insured,Partial Payment تم السداد - جزئي,سداد على دفعتين عن طريق مدلين,MAJED,نوع الرخصه لاتخول,NaN,NaN,NaN,نوع الرخصة لا يخوله بقيادة المركبة,No legal remarks
2585,ASCIC0689,C/CRO1/2025/PCSL/0020267,30487.50,NaN,30487.50,15244.0,15243.0,NaN,منال فهد بن حسن البليهد,966556020307,1012753255,خالد البليهد,9.665560e+11,1.124237e+09,Individual,2025-10-30 00:00:00,25-11-06,Al Jawf,third_party,Partial Payment تم السداد - جزئي,عن طريق مدلين لتحصيل الديون,MAJED,تجاوز الاشارة الحمراء,NaN,NaN,NaN,NaN,2026-05-03: مبدأ الحلول (by Muteb Alotaibi)
2825,ASCIC0436,C/CRO1/2023/PCRA/0000401,156101.40,NaN,156101.40,136101.0,20000.0,1,زياد مسفر الزهراني,966532533555,1124972215,NaN,NaN,NaN,Individual,2023-04-12 00:00:00,25-10-09,Dhahran,third_party,Partial Payment تم السداد - جزئي,حسب طلب العميل,Madeline,ا يملك تأمين,other أخرى,حسب التواصل مع مدلين العميل سدد دفعة 20000 ريا...,"14 Jul, 2026 | 10:42 AM",NaN,2026-05-06: لا يوجد حق رجوع. (by Hallayil ALsh...
2829,ASCIC0440,C/CRO1/2025/PTRA/0014687,3040.50,NaN,3040.50,2040.0,1000.0,NaN,مجاهد ناصر محمد عمر العبدلي,966536998095,2583944117,NaN,NaN,NaN,Individual,2025-09-19 00:00:00,25-10-09,Al Ta'if,insured,Partial Payment تم السداد - جزئي,حسب طلب العميل,MAJED,لا يملك رخصة,NaN,NaN,NaN,لا يملك رخصة قيادة,"2026-05-06: ""تم التبليغ / وفي صدد رفع دعوى (by..."
3174,ASCIC096,C/CRO1/2025/PCRA/0001430,6460.60,NaN,6460.60,3054.0,3406.0,1,علي سالم معجب السبيعي,966509096929,1043879293,NaN,NaN,NaN,Individual,2025-05-21 00:00:00,25-07-10,Riyadh,third_party,Partial Payment تم السداد - جزئي,NaN,MAJED,"التقدير 6,288.10 ريال + رسوم تقدير 172.5 ريال",NaN,NaN,NaN,NaN,No legal remarks


In [18]:
#rule 4: to check if negative money values exist in the dataset
money_cols = [
    "Original Amount",
    "Recovery Amount",
    "Collected Amount",
    "Remaining Amount"
]

(df[money_cols] < 0).sum()

Original Amount     0
Recovery Amount     0
Collected Amount    0
Remaining Amount    0
dtype: int64

## Outliers

In [19]:
import plotly.express as px

fig = px.box(
    df,
    y="Recovery Amount",
    title="Recovery Amount Distribution",
    log_y=True
)

fig.show()

In [20]:
import plotly.express as px

fig = px.box(
    df,
    y="Original Amount",
    title="Original Amount Distribution",
    log_y=True
)

fig.show()

In [21]:
import plotly.express as px

fig = px.box(
    df,
    y="Remaining Amount",
    title="Remaining Amount Distribution",
    log_y=True
)

fig.show()

In [22]:
import plotly.express as px

fig = px.box(
    df,
    y="Collected Amount",
    title="Collected Amount Distribution",
    log_y=True
)

fig.show()

# Summary of Findings

The data quality audit was performed to assess the reliability of the insurance recovery dataset before analysis and dashboard development.

### Key Findings

- The dataset contains **3,267 insurance recovery records** across **28 columns**.
- Key financial fields (Original Amount, Recovery Amount, Remaining Amount, and Collected Amount) are complete with no missing values.
- Date fields were successfully converted to the appropriate datetime format.
- Several columns contain substantial missing values (e.g., Approved Discount %, Driver Information, Latest Remark), which appear to be business-related rather than random missing data.
- No duplicate Claim IDs were found, indicating each claim record has a unique identifier.
- Duplicate Claim Numbers were identified and require business validation to determine whether they represent multiple recovery events or duplicate entries.
- Financial business rules were validated to identify any inconsistencies in monetary values.
- Boxplot analysis indicates the presence of outliers in financial columns. These may represent high-value insurance claims rather than erroneous records and should be investigated before removal.

# Recommendations

Based on the data quality assessment, the following actions are recommended before advanced analysis:

1. Confirm the business meaning of duplicate Claim Numbers before any deduplication.
2. Investigate columns with high missing percentages (e.g., Approved Discount %, Driver Information) to determine whether missing values are expected.
3. Remove or ignore export artifact columns (Unnamed: 24 and Unnamed: 25) after confirming they contain no useful information.
4. Validate financial records that violate business rules with domain experts.
5. Retain detected outliers unless confirmed as data entry errors, as they may represent legitimate high-value claims.
6. Use the cleaned dataset as the foundation for exploratory analysis and dashboard development.